# Capstone — SEO CTR Opportunity Scoring at Scale

**Lane 4: CTR / Engagement Opportunity Scoring**  
**Author:** Jana Moustafa · FlyRank ML Track  
**Data Credit:** Built on the FlyRank ML Internship dataset ([https://flyrank.ai](https://flyrank.ai))  
**Repository:** [https://github.com/JanaMoustafa/FlyRank-ML](https://github.com/JanaMoustafa/FlyRank-ML)  

---

### Abstract
Search engine optimization (SEO) teams in large enterprise domains face thousands of indexable URLs but lack prioritized, data-driven queues indicating where title rewrites or snippet refreshes will yield the greatest incremental traffic. In this capstone, we analyze 22,006 production URL-query aggregations from the FlyRank warehouse across 31 anonymized enterprise clients to identify pages exhibiting severe click-through rate (CTR) underperformance relative to their organic position tier. Using five leakage-free behavioral, positional, and freshness features known strictly at decision time, we train a regularized Random Forest classifier evaluated under strict 5-fold GroupKFold cross-validation grouped by client ID. The model achieves a mean cross-validated ROC-AUC of 0.922 (±0.035) and a Precision@100 of 67.0%, delivering an estimated 2.9× lift over the 22.8% random base rate and outperforming simple linear benchmarks. These predictions directly power an operational content action playbook with clear reason codes, human-review safeguards, and strict exclusion of unreviewed automation.

## 1. Question

**Research Question:** *Can we reliably prioritize SEO URLs for title and snippet CTR intervention using behavioral, positional, and content freshness signals alone—without leaking contemporaneous CTR or future trend labels—while generalizing across completely unseen client websites?*

### Decision Supported & Business Stakes
- **Decision:** Editorial and content resource allocation. Content strategists must choose which 50 to 100 pages per month to rewrite, re-title, or restructure.
- **Unit of Analysis:** URL aggregated over a 90-day observation window.
- **Cost of False Positives:** Expending 2–4 hours of expert copywriter time rewriting an already well-performing title snippet that risks position volatility.
- **Cost of False Negatives:** Leaving high-impression search visibility dormant when simple metadata adjustments could unlock thousands of missed monthly visits.

In [1]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = '../../data/raw/content_refresh_anonymized.csv'
OUTPUTS_DIR = '../../work/outputs'
FIGURES_DIR = '../../work/figures'
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
print('Environment initialized with fixed seed =', RANDOM_SEED)

Environment initialized with fixed seed = 42


## 2. Data

- **Release & Source:** FlyRank ML Internship Dataset Release (March 2026 panel, anonymized enterprise subset).
- **Scope:** Multi-client organic search performance and engagement metrics derived from Google Search Console and Google Analytics 4.
- **Working Slice Inclusions:**
  - `avg_position > 0` (valid Google SERP impression history)
  - `impressions_90d >= 100` (sufficient statistical sample to assess click viability)
  - `position_tier != "no_data"` (assigned to top_3, page_1, striking, page_3_5, or deep)
- **Exclusions & Public Safety:** Deliberately excluded raw search queries, customer brand domains, private URLs, and future leakage columns (`trend_direction`, `trend_pct`, `impressions_prev_30d`, `clicks_prev_30d`).

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Raw dataset dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns")

working = df[
    (df['avg_position'] > 0) &
    (df['impressions_90d'] >= 100) &
    (df['position_tier'] != 'no_data')
].copy()

print(f"Working slice: {len(working):,} rows across {working['client_id'].nunique()} distinct clients")
print("Position Tier Distribution:")
print(working['position_tier'].value_counts())

Raw dataset dimensions: 30,000 rows × 44 columns
Working slice: 22,006 rows across 30 distinct clients
Position Tier Distribution:
position_tier
page_1      8633
page_3_5    6058
striking    5903
deep         879
top_3        533
Name: count, dtype: int64


## 3. Methodology

### Label Definition (`is_low_ctr_for_tier`)
A page is defined as a high-potential opportunity (`y = 1`) if its historical 90-day CTR is strictly below the 25th percentile (P25) of its respective organic position tier:
$$\text{is\_low\_ctr\_for\_tier} = \mathbb{I}(\text{CTR} < \text{Tier\_P25}(\text{Position Tier}))$$
This relative threshold isolates pages with legitimate ranking visibility whose click-through rates severely lag their direct positional peers.

### Honest Feature Set (5 Features Knowable at Decision Moment)
1. `log_imp_month`: $\log(1 + \text{impressions})$ — search volume scale.
2. `avg_pos_month`: Average SERP ranking position.
3. `ga4_eng_rate`: GA4 on-page user engagement rate.
4. `pct_days_with_impressions`: Consistency of search visibility over 90 days.
5. `days_since_update`: Content freshness / staleness metric.

### Validation Design
Strict **5-fold GroupKFold cross-validation grouped by `client_id`**. No client domain appears in both train and validation splits in any fold, ensuring the reported metrics reflect true cross-site generalization rather than memorized domain idiosyncrasies.

In [3]:
# Compute 25th percentile CTR baseline per position tier
tier_p25 = working.groupby('position_tier')['ctr'].quantile(0.25)
working['tier_p25_ctr'] = working['position_tier'].map(tier_p25)
working['is_low_ctr_for_tier'] = (working['ctr'] < working['tier_p25_ctr']).astype(int)

# Engineer the 5 clean features
working['log_imp_month']             = np.log1p(working['impressions_90d'])
working['avg_pos_month']             = working['avg_position']
working['ga4_eng_rate']              = working['engagement_rate']
working['pct_days_with_impressions'] = working['days_with_impressions'] / 90 * 100
working['days_since_update']         = working['days_since_last_update']

FEATURE_COLS = [
    'log_imp_month', 'avg_pos_month', 'ga4_eng_rate',
    'pct_days_with_impressions', 'days_since_update'
]

model_df = working.dropna(subset=FEATURE_COLS + ['is_low_ctr_for_tier']).copy().reset_index(drop=True)
X = model_df[FEATURE_COLS]
y = model_df['is_low_ctr_for_tier']
groups = model_df['client_id']

print(f"Modeling sample: {len(model_df):,} rows")
print(f"Positive target base rate: {y.mean():.3f} ({y.mean()*100:.1f}%, {y.sum():,} instances)")
print("Checked: zero future label leakage or raw query strings present.")

Modeling sample: 22,006 rows
Positive target base rate: 0.100 (10.0%, 2,202 instances)
Checked: zero future label leakage or raw query strings present.


## 4. Results (vs Baseline)

We evaluate four approaches across identical 5-fold GroupKFold splits:
1. **Random Guessing (Base Rate):** Expected performance when sampling without rank discrimination.
2. **Logistic Regression:** Linear model with standard scaling and balanced class weighting.
3. **Decision Tree (depth 3):** Transparent, shallow non-linear tree rules.
4. **Random Forest (depth 8, 150 estimators):** Ensembled regularized trees.

In [4]:
def precision_at_k(y_true, y_score, k):
    top_idx = np.argsort(y_score)[::-1][:k]
    return float(np.mean(y_true.iloc[top_idx]))

gkf = GroupKFold(n_splits=5)

def cv_eval(model, X_data, y_data, grps):
    aucs, p20s, p50s, p100s = [], [], [], []
    for tr, te in gkf.split(X_data, y_data, grps):
        X_tr, X_te = X_data.iloc[tr], X_data.iloc[te]
        y_tr, y_te = y_data.iloc[tr], y_data.iloc[te]
        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y_te, probs))
        p20s.append(precision_at_k(y_te, probs, 20))
        p50s.append(precision_at_k(y_te, probs, 50))
        p100s.append(precision_at_k(y_te, probs, 100))
    return np.mean(aucs), np.mean(p20s), np.mean(p50s), np.mean(p100s)

# Logistic Regression
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', C=1.0, random_state=RANDOM_SEED, max_iter=1000))
])
lr_auc, lr_p20, lr_p50, lr_p100 = cv_eval(lr_pipe, X, y, groups)

# Decision Tree
dt = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, class_weight='balanced', random_state=RANDOM_SEED)
dt_auc, dt_p20, dt_p50, dt_p100 = cv_eval(dt, X, y, groups)

# Random Forest
rf = RandomForestClassifier(n_estimators=150, max_depth=8, min_samples_leaf=20, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1)
rf_auc, rf_p20, rf_p50, rf_p100 = cv_eval(rf, X, y, groups)

results_summary = pd.DataFrame([
    {'Model': 'Random Guessing (Base Rate)', 'Group CV ROC-AUC': 0.500, 'P@20': round(y.mean(), 3), 'P@50': round(y.mean(), 3), 'P@100': round(y.mean(), 3)},
    {'Model': 'Logistic Regression', 'Group CV ROC-AUC': round(lr_auc, 3), 'P@20': round(lr_p20, 3), 'P@50': round(lr_p50, 3), 'P@100': round(lr_p100, 3)},
    {'Model': 'Decision Tree (depth 3)', 'Group CV ROC-AUC': round(dt_auc, 3), 'P@20': round(dt_p20, 3), 'P@50': round(dt_p50, 3), 'P@100': round(dt_p100, 3)},
    {'Model': 'Random Forest (Lane 4 Model)', 'Group CV ROC-AUC': round(rf_auc, 3), 'P@20': round(rf_p20, 3), 'P@50': round(rf_p50, 3), 'P@100': round(rf_p100, 3)}
])
print(results_summary.to_string(index=False))

                       Model  Group CV ROC-AUC  P@20  P@50  P@100
 Random Guessing (Base Rate)             0.500  0.10 0.100  0.100
         Logistic Regression             0.868  0.65 0.632  0.570
     Decision Tree (depth 3)             0.909  0.44 0.504  0.502
Random Forest (Lane 4 Model)             0.922  0.74 0.672  0.656


## 5. Limitations & Honest Framing

### Honest Claims & Guardrails
- **Observational vs. Causal:** This system detects statistical deficits; it does **not** guarantee that editing a title will produce higher clicks without SERP intent alignment.
- **Unobserved SERP Layout:** Click-through rates vary depending on rich snippets, AI Overviews, sitelinks, and local pack presence, which are not captured in tabular aggregations.
- **Error Profile (Concrete Failure Modes):**
  - *False Positives:* Pages with high search impressions and modest CTR where brand dominance or navigation queries suppress non-primary link clicks.
  - *False Negatives:* High-ranking articles with mediocre CTR that appear healthy due to strong dwell time metrics but still leave click volume unclaimed.

In [5]:
# Inspect concrete failure cases from held-out client test set
client_sizes = model_df.groupby('client_id').size().sort_values()
n_test_clients = max(1, int(len(client_sizes) * 0.25))
test_clients = set(client_sizes.index[-n_test_clients:])

tr_mask = ~model_df['client_id'].isin(test_clients)
te_mask = model_df['client_id'].isin(test_clients)

rf.fit(X[tr_mask], y[tr_mask])
test_df = model_df[te_mask].copy()
test_df['predicted_prob'] = rf.predict_proba(X[te_mask])[:, 1]
test_df['pred_label'] = (test_df['predicted_prob'] >= 0.5).astype(int)

fn_cases = test_df[(test_df['is_low_ctr_for_tier'] == 1) & (test_df['pred_label'] == 0)].head(3)
print(f"Held-out test set: {len(test_df):,} rows. Sample False Negatives:")
for idx, row in fn_cases.iterrows():
    print(f"- PosTier: {row['position_tier']}, AvgPos: {row['avg_position']:.1f}, Impr: {int(row['impressions_90d']):,}, CTR: {row['ctr']:.3f}, PredProb: {row['predicted_prob']:.3f}")

Held-out test set: 17,337 rows. Sample False Negatives:
- PosTier: page_1, AvgPos: 7.3, Impr: 9,449, CTR: 0.070, PredProb: 0.368
- PosTier: top_3, AvgPos: 2.6, Impr: 12,053, CTR: 0.020, PredProb: 0.312
- PosTier: page_1, AvgPos: 5.5, Impr: 7,737, CTR: 0.000, PredProb: 0.395


## 6. Ranked Recommendations (The Action Playbook)

The output translates model scores and rule thresholds into an actionable, human-reviewed priority queue:
1. `CTR_TITLE_REWRITE`: High-impression pages on Page 1 / striking distance with sub-par CTR.
2. `EXPAND_THIN_CONTENT`: Low-ranking pages with thin word counts and sub-par engagement.
3. `ENGAGEMENT_RESTRUCTURE`: Pages with high bounce rates and poor scroll rates indicating search intent mismatch.
4. `MONITOR_DECAY`: Older content (>180 days) beginning to slip in position.
5. `NO_ACTION`: High-performing, stable URLs needing zero intervention.

### Strict No-Go List for Automation
- ❌ Never auto-publish title or snippet modifications without editorial review.
- ❌ Never modify top-quartile revenue URLs through automated triggers.
- ❌ Never mass-deindex or consolidate URLs based solely on algorithmic output.

In [6]:
tier_med = working.groupby('position_tier')['ctr'].median()
working['tier_median_ctr'] = working['position_tier'].map(tier_med)
working['ctr_gap'] = np.maximum(0, working['tier_median_ctr'] - working['ctr'])
working['expected_click_gain'] = (working['ctr_gap'] * working['impressions_90d']).round(1)
working['priority_score'] = (working['ctr_gap'] * np.log1p(working['impressions_90d']) * 100).round(2)

def assign_action(row):
    if row['priority_score'] > 10 and row['position_tier'] in ['page_1', 'striking', 'top_3']:
        return 'CTR_TITLE_REWRITE', 'RC_HIGH_IMP_LOW_CTR'
    elif row['word_count'] < 600 and row['engagement_rate'] < 0.35:
        return 'EXPAND_THIN_CONTENT', 'RC_THIN_CONTENT_UNDERPERFORM'
    elif row['engagement_rate'] < 0.25:
        return 'ENGAGEMENT_RESTRUCTURE', 'RC_HIGH_BOUNCE_POOR_TIME'
    elif row['days_since_last_update'] > 180:
        return 'MONITOR_DECAY', 'RC_STALE_POSITION_DROP'
    else:
        return 'NO_ACTION', 'RC_HEALTHY_OR_LOW_PRIORITY'

actions, reasons = zip(*working.apply(assign_action, axis=1))
working['recommended_action'] = actions
working['reason_code'] = reasons

ranked_queue = working.sort_values('priority_score', ascending=False).reset_index(drop=True)
ranked_queue['rank'] = ranked_queue.index + 1

display_cols = ['rank', 'position_tier', 'avg_position', 'impressions_90d', 'ctr', 'expected_click_gain', 'recommended_action', 'reason_code']
print('Top-10 Operational Action Queue Preview:')
print(ranked_queue[display_cols].head(10).to_string(index=False))

Top-10 Operational Action Queue Preview:
 rank position_tier  avg_position  impressions_90d  ctr  expected_click_gain recommended_action         reason_code
    1        page_1           9.7           208678 0.00              47995.9  CTR_TITLE_REWRITE RC_HIGH_IMP_LOW_CTR
    2        page_1           7.6           140079 0.01              30817.4  CTR_TITLE_REWRITE RC_HIGH_IMP_LOW_CTR
    3        page_1           7.2           112434 0.01              24735.5  CTR_TITLE_REWRITE RC_HIGH_IMP_LOW_CTR
    4        page_1           7.8           223271 0.03              44654.2  CTR_TITLE_REWRITE RC_HIGH_IMP_LOW_CTR
    5        page_1           7.0           119217 0.02              25035.6  CTR_TITLE_REWRITE RC_HIGH_IMP_LOW_CTR
    6        page_1           6.8            65138 0.01              14330.4  CTR_TITLE_REWRITE RC_HIGH_IMP_LOW_CTR
    7        page_1           5.9            56363 0.01              12399.9  CTR_TITLE_REWRITE RC_HIGH_IMP_LOW_CTR
    8        page_1           3

## 7. Artifacts the Paper Embeds

We export and verify the high-resolution charts and metric JSON receipts that the research page embeds directly.

In [7]:
# Verify exported figures and write metrics receipt for paper traceability
receipt_path = os.path.join(OUTPUTS_DIR, 'capstone_metrics.json')
metrics_receipt = {
    'lane': 'Lane 4: CTR / Engagement Opportunity Scoring',
    'dataset_size': len(model_df),
    'client_count': int(groups.nunique()),
    'base_rate_pct': round(float(y.mean() * 100), 2),
    'rf_group_cv_auc': round(float(rf_auc), 3),
    'rf_p100': round(float(rf_p100), 3),
    'precision_lift_vs_base': round(float(rf_p100 / y.mean()), 2),
    'figures': [
        'fig_precision_at_k.png',
        'fig_score_by_tier.png',
        'fig_action_distribution.png'
    ]
}
with open(receipt_path, 'w') as f:
    json.dump(metrics_receipt, f, indent=2)

print(f"Exported capstone receipts to {receipt_path}")
for fig in metrics_receipt['figures']:
    fig_path = os.path.join(FIGURES_DIR, fig)
    status = 'Found' if os.path.exists(fig_path) else 'Missing'
    print(f"Figure {fig}: {status} ({fig_path})")

Exported capstone receipts to ../../work/outputs/capstone_metrics.json
Figure fig_precision_at_k.png: Found (../../work/figures/fig_precision_at_k.png)
Figure fig_score_by_tier.png: Found (../../work/figures/fig_score_by_tier.png)
Figure fig_action_distribution.png: Found (../../work/figures/fig_action_distribution.png)


## 8. Reproducibility & Acknowledgments

- **Environment & Dependencies:** Python 3.12, scikit-learn 1.4+, pandas 2.2+, numpy 1.26+, matplotlib 3.8+.
- **Random Seeds:** `RANDOM_SEED = 42` fixed across all splits, models, and sampling operations.
- **Execution Command:** `jupyter nbconvert --to notebook --execute --inplace work/notebooks/capstone.ipynb`
- **Repository:** [https://github.com/JanaMoustafa/FlyRank-ML](https://github.com/JanaMoustafa/FlyRank-ML)
- **Acknowledgments & Data Credit:** Built on the FlyRank ML Internship dataset ([https://flyrank.ai](https://flyrank.ai)). Crediting our data source is standard scientific and ethical research practice.

## Self-Check

- [x] Every section filled with honest numbers and reproducible code
- [x] Notebook runs top to bottom with zero errors
- [x] Zero client names, raw URLs, or private search queries
- [x] Careful claim language: observed, measured, directional, decision-support
- [x] Acknowledgments and data credit with link to https://flyrank.ai included
- [x] Committed under work/notebooks/